# INDEX
0) [**INDEX**](#index)
1) [**IMPORTS**](#imports)
2) [**GAME LOGIC**](#game-logic)
    + [GameState](#class-gamestate)
        + [Play Human v Human](#testing-game) 
3) [**MCTS**](#mcts)
    + [Baseline MCTS](#baseline-mcts)
        + [Test Baseline MCTS](#testing-baseline-mcts)
    + [Easy MCTS](#easy-mcts)
        + [Test eMCTS](#testing-easy-mcts)
4) [**DECISION TREE**](#decision-tree)

In [ ]:
from collections import deque
from copy import deepcopy
from IPython.display import clear_output  # pyright: ignore[reportMissingImports]
import math
import random

# GAME LOGIC

## class GameState
### Variables:
```py
    h: int
    w: int
    board: list[deque]
    player: int
    prevStates: dict
    # winner: (int | None) -- Useless
    gameOver: bool
```
### Methods:
```py
    # getCopy(self: Self@GameState) -> Self@GameState -- Useless
    isFull(self: Self@GameState) -> bool
    getWinner(self: Self@GameState) -> (int | None)
    checkRepetition(self: Self@GameState) -> bool
    getValid(self: Self@GameState) -> list
    move(
        self: Self@GameState,
        col: int,
        t: str
    ) -> bool
    printB(self: Self@GameState) -> None
```


In [ ]:
class GameState:
    def __init__(self):
        self.h=6
        self.w=7
        self.board=[deque(maxlen=self.h) for _ in range(self.w)]
        self.player=1 # X=1; O=2
        self.prevStates = {}
        self.winner=None # Useless, same as getWinner()
        self.gameOver=False

    def getCopy(self):
        """Returns a copy of the current state"""
        return deepcopy(self)

    def isFull(self):
        """Check if Board is full"""
        return all(len(col) == self.h for col in self.board)

    def getWinner(self):
        """Checks for winner, returns None if no winner exists"""
    
        def check_direction(x, y, dx, dy):
            start = self.board[x][y]
            if start == 0:
                return None
            for i in range(1, 4):
                nx = x + dx * i
                ny = y + dy * i

                if nx < 0 or nx >= self.w or ny < 0 or ny >= len(self.board[nx]):
                    return None

                if self.board[nx][ny] != start:
                    return None

            return start

        winners = set()

        for x in range(self.w):
            for y in range(len(self.board[x])):

                for dx, dy in [(1,0), (0,1), (1,1), (1,-1)]:
                    result = check_direction(x, y, dx, dy)
                    if result is not None:
                        winners.add(result)

        if len(winners) == 1:
            return winners.pop()
        elif len(winners) > 1:
            return self.player
        else:
            return None

    def checkRepetition(self):
        """Check if the position has been reached 3 times"""
        s = tuple(tuple(col) for col in self.board) # Tuples are comparable
        if s not in self.prevStates:
            self.prevStates[s]=0
        self.prevStates[s]+=1

        return self.prevStates[s] >= 3

    def getValid(self):
        """Returns list of valid moves"""

        out=[]
        if self.gameOver:
            return out
        for c in range(self.w):
            col=self.board[c]
            if len(col) > 0 and col[0] == self.player:
                out.append((c, 'p'))
            if len(col) < self.h:
                out.append((c, 'i'))
        return out

    def move(self,col,t): # t = 'i' "insert" | 'p' "pop"
        """Handles move of type t in column col"""

        lm = self.getValid()
        if len(lm) == 0:
            self.gameOver=True
            return False
        if (col, t) not in lm:
            # Draw on insert to full board to avoid getting stuck on HvH games,
            # this part should be unreachable now because of the previous if-then
            if t == 'i' and 0 <= col < self.w and self.isFull():
                self.gameOver=True
            return False

        if t == 'i':
            self.board[col].append(self.player)
        else:
            self.board[col].popleft()

        winner = self.getWinner()
        if winner:
            self.gameOver=True
            self.winner=winner
        if self.isFull():
            self.gameOver=True
        if self.checkRepetition():
            self.gameOver=True

        #if not self.gameOver: -- Removed: Caused MCTS to never play winning moves
        self.player=3-self.player
        return True

    def printB(self):
        """Print the board (ASCII) to standard output"""
        print("+-------+")

        for row in range(self.h - 1, -1, -1):
            print("|", end="")
            for col in range(self.w):
                if row >= len(self.board[col]):
                    print(" ", end="")
                elif self.board[col][row] == 1:
                    print("X", end="")
                elif self.board[col][row] == 2:
                    print("O", end="")
                else:
                    print(" ", end="")
            print("|")

        print("+-------+")
        print("|0123456|")
        print("+-------+")

        return None

### Testing game: 
Run to play the game Human vs Human.  
The input is done on separate lines with a single character `p` (for pop) or `i` (for insert) in the first line followed by an integer representing the column number to play on in the second line.

In [ ]:
g=GameState()

g.printB()
while not g.gameOver:
    t=input("Type: ") # 'i' <- insert | 'p' <- pop
    c=int(input("Column: "))

    if not g.move(c,t):
        print("Invalid Move")
    else:
        clear_output()
        g.printB()

print("--- Winner: ",g.winner,"---")

# MCTS

## Baseline MCTS

In [ ]:
class MCTSNode:
    def __init__(self, state: GameState, parent=None, action=None):
        self.state=state
        self.parent= parent
        self.action=action
        if parent is None:
            self.player=state.player
        else:
            self.player=3-state.player
        self.children=[]
        self.visits=0
        self.wins=0.0
        self.untriedActions=state.getValid()

    def isTerminal(self):
        """Check if node is terminal"""
        return self.state.gameOver
    
    def fullyExpanded(self):
        """Check if all actions are explored"""
        return len(self.untriedActions) == 0
    
    def expand(self):
        """Expand node"""
        (c,t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(c,t)

        child = MCTSNode(newState,parent=self,action=(c,t))
        self.children.append(child)
        return child

    def selectChild(self,c=1.4):
        for child in self.children:
            if child.visits == 0:
                return child
        
        def ucb(child: MCTSNode):
            exploit = child.wins / child.visits
            explore = c * math.sqrt(math.log(self.visits) / child.visits)
            return exploit + explore
    
        return max (self.children, key=ucb)

    def rollout(self):
        """Plays random moves until the game finishes"""
        state = deepcopy(self.state)
        while True:
            if state.gameOver:
                return state.winner
            (c,t) = random.choice(state.getValid())
            state.move(c,t)

    def backpropagate(self,winner):
        """Updates the wins and visits from simulation results"""
        self.visits+=1

        if winner is None:
            self.wins += 0.5
        elif winner == self.player:
            self.wins += 1.0

        if self.parent:
            self.parent.backpropagate(winner)

        return

# -- end of class --

def mctsSearch(rootState: GameState, iterations = 500):
    root = MCTSNode(rootState)
    for _ in range (iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node=node.expand()
        winner = node.rollout()
        node.backpropagate(winner)

    best = max(root.children,key=lambda c: c.visits)
    return best.action

### Testing baseline MCTS
Run to play the game Human vs MCTS.  
Change the line `if g.player == 1:` into `if g.player == 2:` to play second.  

The input is done on separate lines with a single character `p` (for pop) or `i` (for insert) in the first line followed by an integer representing the column number to play on in the second line.

In [ ]:
g=GameState()

g.printB()
while not g.gameOver:
    if g.player == 1: # Chage to 2 to let the algorithm start first
        t=input("Type: ") # 'i' <- insert | 'p' <- pop
        c=int(input("Column: "))

        if not g.move(c,t):
            print("Invalid Move")
        else:
            clear_output()
            g.printB()
    else:
        (c,t) = mctsSearch(g,iterations=1000)

        g.move(c,t)
        clear_output()
        g.printB()

print("--- Winner:",g.winner,"---")

## Easy MCTS
Easier difficulty implemented by sometimes playing a random move instead of trying to find the best strategy

In [ ]:
def mctstupidSearch(rootState: GameState, iterations = 500):
    if random.random() < 0.25:
        moves = rootState.getValid()
        #print("Oops!") # See how often it misses during training
        return random.choice(moves)
    return mctsSearch(rootState,iterations=iterations)

### Testing Easy MCTS

#### Human vs eMCTS
Run to play the game Human vs eMCTS.  
Change the line `if g.player == 1:` into `if g.player == 2:` to play second.  

The input is done on separate lines with a single character `p` (for pop) or `i` (for insert) in the first line followed by an integer representing the column number to play on in the second line.

In [ ]:
g=GameState()

g.printB()
while not g.gameOver:
    if g.player == 1: # Chage to 2 to let the algorithm start first
        t=input("Type: ") # 'i' <- insert | 'p' <- pop
        c=int(input("Column: "))

        if not g.move(c,t):
            print("Invalid Move")
        else:
            clear_output()
            g.printB()
    else:
        (c,t) = mctstupidSearch(g,iterations=1000)

        g.move(c,t)
        clear_output()
        g.printB()

print("--- Winner:",g.winner,"---")

#### eMCTS vs MCTS
Run to play 10 games of eMCTS vs baseline MCTS.  
Change the line `if g.player == 1:` into `if g.player == 2:` to switch their order of play.  

In [ ]:
for _ in range (10):
    g=GameState()

    #g.printB()
    while not g.gameOver:
        if g.player == 1:
            (c,t) = mctsSearch(g,iterations=1000)

            g.move(c,t)
            #clear_output()
            #g.printB()
        else:
            (c,t) = mctsSearch(g,iterations=1000)

            g.move(c,t)
            #clear_output()
            #g.printB()

    print("--- Winner:",g.winner,"---")

# DECISION TREE

In [ ]:
# ------
#  TODO
# ------